# Laboratório — Forward pass vetorizado e cache de intermediários

Implementaremos uma MLP em **NumPy puro**, com exemplos nas linhas, validação estrita de
shapes e cache explícito por camada. O objetivo termina nos logits: não há loss,
backpropagation, autograd nem atualização de parâmetros nesta aula.

Este notebook é o artefato executável da Aula 05 do M5 — Redes Neurais do Zero.


## Goal

Ao final, teremos evidência de que:

1. o exemplo manual e a implementação coincidem;
2. o forward vetorizado equivale ao processamento linha a linha;
3. cada exemplo permanece independente e a ordem das linhas é preservada;
4. shapes ambíguos são rejeitados antes da multiplicação;
5. o cache contém exatamente os intermediários declarados;
6. repetir o forward não altera entradas ou parâmetros;
7. a memória dos intermediários cresce linearmente com o lote;
8. referências mutáveis exigem um contrato de não mutação.


## Setup

- Python >= 3.11
- NumPy >= 1.26
- Matplotlib >= 3.8
- nbformat >= 5.9 apenas para validar o arquivo

Dados: matrizes pequenas explícitas e geração sintética local. Não há download,
credenciais, dataset externo ou estado oculto. Seed: `20260909`; dtype: `float64`.


In [ ]:
from dataclasses import dataclass
import platform

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260909
DTYPE = np.float64
rng = np.random.default_rng(SEED)

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "matplotlib": plt.matplotlib.__version__,
    "seed": SEED,
    "dtype": str(np.dtype(DTYPE)),
})


## Steps

### 1. Ativações locais

Reutilizamos as funções da Aula 04. `identity` preserva logits; sigmoid usa ramos para
evitar formar uma exponencial positiva extrema. Todas preservam o shape recebido.


In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=DTYPE)
    out = np.empty_like(z)
    positive = z >= 0.0
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    out[~positive] = exp_z / (1.0 + exp_z)
    return out


def activation_forward(z: np.ndarray, name: str) -> np.ndarray:
    functions = {
        "identity": lambda value: value.copy(),
        "relu": lambda value: np.maximum(value, 0.0),
        "tanh": np.tanh,
        "sigmoid": sigmoid,
    }
    if name not in functions:
        raise ValueError(f"ativação desconhecida: {name!r}")
    result = np.asarray(functions[name](z), dtype=DTYPE)
    if result.shape != z.shape:
        raise RuntimeError("a ativação alterou o shape")
    if not np.isfinite(result).all():
        raise FloatingPointError("a ativação produziu valor não finito")
    return result


### 2. Contratos de shape antes do cálculo

Uma amostra continua 2D: `(1, d_in)`. Pesos usam `(d_in, d_out)` e o viés usa
exatamente `(d_out,)`. Não aceitamos `(1, d_out)` mesmo que o NumPy pudesse transmiti-lo.


In [ ]:
class ShapeContractError(ValueError):
    pass  # erro de dimensão próximo à causa, antes do produto matricial


def as_batch(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=DTYPE)
    if x.ndim != 2:
        raise ShapeContractError(f"X deve ser 2D (n, d); recebido {x.shape}")
    if x.shape[0] < 1 or x.shape[1] < 1:
        raise ShapeContractError("X precisa ter pelo menos uma linha e uma coluna")
    if not np.isfinite(x).all():
        raise FloatingPointError("X contém valor não finito")
    return x


def validate_layer_shapes(a_prev: np.ndarray, w: np.ndarray, b: np.ndarray) -> None:
    if a_prev.ndim != 2:
        raise ShapeContractError(f"a_prev deve ser 2D; recebido {a_prev.shape}")
    if w.ndim != 2:
        raise ShapeContractError(f"W deve ser 2D; recebido {w.shape}")
    if b.ndim != 1:
        raise ShapeContractError(f"b deve ser 1D; recebido {b.shape}")
    if a_prev.shape[1] != w.shape[0]:
        raise ShapeContractError(
            f"eixos internos incompatíveis: {a_prev.shape} @ {w.shape}"
        )
    expected_bias = (w.shape[1],)
    if b.shape != expected_bias:
        raise ShapeContractError(f"b deveria ter shape {expected_bias}; recebeu {b.shape}")
    if not all(np.isfinite(array).all() for array in (a_prev, w, b)):
        raise FloatingPointError("entrada ou parâmetro contém valor não finito")


### 3. Etapa afim

Com exemplos nas linhas, `a_prev @ w` contrai o eixo `d_in`. O viés 1D é somado a
cada linha por broadcasting. O shape esperado é verificado também na saída.


In [ ]:
def affine_forward(a_prev: np.ndarray, w: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_prev = np.asarray(a_prev, dtype=DTYPE)
    w = np.asarray(w, dtype=DTYPE)
    b = np.asarray(b, dtype=DTYPE)
    validate_layer_shapes(a_prev, w, b)
    z = a_prev @ w + b
    expected = (a_prev.shape[0], w.shape[1])
    if z.shape != expected:
        raise RuntimeError(f"saída afim deveria ser {expected}; recebeu {z.shape}")
    if not np.isfinite(z).all():
        raise FloatingPointError("etapa afim produziu valor não finito")
    return z


### 4. Cache estruturado da camada densa

O cache registra `a_prev`, `w`, `b`, `z` e o nome da ativação. `frozen=True` impede
reatribuir campos, mas não congela o conteúdo dos arrays; a não mutação é um contrato.


In [ ]:
@dataclass(frozen=True)
class DenseCache:
    a_prev: np.ndarray
    w: np.ndarray
    b: np.ndarray
    z: np.ndarray
    activation: str


def dense_forward(
    a_prev: np.ndarray,
    w: np.ndarray,
    b: np.ndarray,
    activation: str,
) -> tuple[np.ndarray, DenseCache]:
    a_prev = np.asarray(a_prev, dtype=DTYPE)
    w = np.asarray(w, dtype=DTYPE)
    b = np.asarray(b, dtype=DTYPE)
    z = affine_forward(a_prev, w, b)
    a = activation_forward(z, activation)
    cache = DenseCache(a_prev=a_prev, w=w, b=b, z=z, activation=activation)
    return a, cache


### 5. Exemplo resolvido de duas camadas

Calculamos uma rede `2 → 3 → 2`, com ReLU oculta e identidade na saída. Os valores
esperados foram derivados na aula; a comparação não usa arredondamento intermediário.


In [ ]:
x_manual = np.array([[1.0, -2.0], [0.5, 3.0]], dtype=DTYPE)
w1_manual = np.array([[1.0, -1.0, 0.5], [2.0, 0.0, -1.0]], dtype=DTYPE)
b1_manual = np.array([0.5, -0.5, 1.0], dtype=DTYPE)
w2_manual = np.array([[1.0, -1.0], [0.5, 2.0], [-2.0, 0.25]], dtype=DTYPE)
b2_manual = np.array([0.1, -0.2], dtype=DTYPE)

z1_expected = np.array([[-2.5, -1.5, 3.5], [7.0, -1.0, -1.75]])
a1_expected = np.array([[0.0, 0.0, 3.5], [7.0, 0.0, 0.0]])
logits_expected = np.array([[-6.9, 0.675], [7.1, -7.2]])

a1_manual, cache1_manual = dense_forward(
    x_manual, w1_manual, b1_manual, "relu"
)
logits_manual, cache2_manual = dense_forward(
    a1_manual, w2_manual, b2_manual, "identity"
)

assert np.array_equal(cache1_manual.z, z1_expected)
assert np.array_equal(a1_manual, a1_expected)
assert np.allclose(logits_manual, logits_expected, atol=1e-15, rtol=0.0)
print({"z1": cache1_manual.z.tolist(), "a1": a1_manual.tolist()})
print({"logits": logits_manual.tolist(), "maior_erro": float(np.max(np.abs(logits_manual - logits_expected)))})


### 6. Forward de uma MLP

O laço percorre camadas; cada camada recebe o lote inteiro. Antes do cálculo, validamos
toda a cadeia de larguras para não retornar uma execução parcial.


In [ ]:
def validate_network(
    x: np.ndarray,
    parameters: list[tuple[np.ndarray, np.ndarray]],
    activations: list[str],
) -> np.ndarray:
    x = as_batch(x)
    if not parameters:
        raise ValueError("a rede precisa ter ao menos uma camada")
    if len(parameters) != len(activations):
        raise ValueError("cada camada precisa de uma ativação")
    width = x.shape[1]
    valid_activations = {"identity", "relu", "tanh", "sigmoid"}
    for layer, ((w, b), activation) in enumerate(zip(parameters, activations), start=1):
        w = np.asarray(w)
        b = np.asarray(b)
        probe = np.zeros((1, width), dtype=DTYPE)
        validate_layer_shapes(probe, w, b)
        if activation not in valid_activations:
            raise ValueError(f"ativação desconhecida na camada {layer}: {activation!r}")
        width = w.shape[1]
    return x


def mlp_forward(
    x: np.ndarray,
    parameters: list[tuple[np.ndarray, np.ndarray]],
    activations: list[str],
) -> tuple[np.ndarray, tuple[DenseCache, ...]]:
    a = validate_network(x, parameters, activations)
    caches = []
    for (w, b), activation in zip(parameters, activations):
        a, cache = dense_forward(a, w, b, activation)
        caches.append(cache)
    return a, tuple(caches)


manual_parameters = [(w1_manual, b1_manual), (w2_manual, b2_manual)]
manual_activations = ["relu", "identity"]
manual_output, manual_caches = mlp_forward(
    x_manual, manual_parameters, manual_activations
)
assert np.array_equal(manual_output, logits_manual)
assert len(manual_caches) == 2
print({"output_shape": manual_output.shape, "cache_count": len(manual_caches)})


### 7. Vetorizado versus linha a linha

Criamos uma rede `4 → 7 → 5 → 3` e 17 exemplos. A referência chama o mesmo forward
para lotes unitários e empilha as respostas. Esperamos igualdade até erro de ponto flutuante.


In [ ]:
def make_parameters(widths: list[int], generator: np.random.Generator):
    result = []
    for d_in, d_out in zip(widths[:-1], widths[1:]):
        w = generator.normal(0.0, 0.3, size=(d_in, d_out)).astype(DTYPE)
        b = generator.normal(0.0, 0.1, size=(d_out,)).astype(DTYPE)
        result.append((w, b))
    return result


widths = [4, 7, 5, 3]
parameters = make_parameters(widths, rng)
activations = ["relu", "tanh", "identity"]
x_batch = rng.normal(size=(17, widths[0])).astype(DTYPE)

output_batch, caches_batch = mlp_forward(x_batch, parameters, activations)
output_rows = np.vstack([
    mlp_forward(x_batch[index:index + 1], parameters, activations)[0]
    for index in range(x_batch.shape[0])
])
vectorization_error = float(np.max(np.abs(output_batch - output_rows)))
vectorized_matmul_calls = len(parameters)
rowwise_matmul_calls = x_batch.shape[0] * len(parameters)

assert vectorization_error < 1e-12
assert output_batch.shape == (17, 3)
print({
    "maior_erro_vetorizado_vs_linhas": vectorization_error,
    "matmuls_vetorizado": vectorized_matmul_calls,
    "matmuls_por_linhas": rowwise_matmul_calls,
})


### 8. Independência, permutação e repetibilidade

Estas propriedades valem para a rede atual porque não há operação que agregue linhas.
Também verificamos que o forward não modifica entrada, pesos ou vieses.


In [ ]:
x_before = x_batch.copy()
parameters_before = [(w.copy(), b.copy()) for w, b in parameters]

single_output, _ = mlp_forward(x_batch[6:7], parameters, activations)
single_error = float(np.max(np.abs(single_output[0] - output_batch[6])))

permutation = rng.permutation(x_batch.shape[0])
permuted_output, _ = mlp_forward(x_batch[permutation], parameters, activations)
permutation_error = float(np.max(np.abs(permuted_output - output_batch[permutation])))

repeated_output, _ = mlp_forward(x_batch, parameters, activations)
repeat_error = float(np.max(np.abs(repeated_output - output_batch)))

assert single_error < 1e-12
assert permutation_error < 1e-12
assert repeat_error == 0.0
assert np.array_equal(x_batch, x_before)
assert all(
    np.array_equal(w, w0) and np.array_equal(b, b0)
    for (w, b), (w0, b0) in zip(parameters, parameters_before)
)
print({
    "erro_exemplo_isolado": single_error,
    "erro_permutacao": permutation_error,
    "erro_repeticao": repeat_error,
    "entradas_e_parametros_preservados": True,
})


### 9. Contraprovas de shape e finitude

Testamos falhas que poderiam ser mascaradas por convenções flexíveis: viés 2D, largura
incompatível, amostra 1D, ativação inexistente e entrada infinita.


In [ ]:
def expect_error(function, exception_type) -> str:
    try:
        function()
    except exception_type as error:
        return str(error)
    raise AssertionError(f"era esperado {exception_type.__name__}")


shape_errors = {
    "bias_2d": expect_error(
        lambda: affine_forward(np.ones((2, 3)), np.ones((3, 4)), np.ones((1, 4))),
        ShapeContractError,
    ),
    "inner_axis": expect_error(
        lambda: affine_forward(np.ones((2, 3)), np.ones((2, 4)), np.ones(4)),
        ShapeContractError,
    ),
    "sample_1d": expect_error(
        lambda: mlp_forward(np.ones(4), parameters, activations),
        ShapeContractError,
    ),
    "activation": expect_error(
        lambda: mlp_forward(x_batch, parameters, ["relu", "mystery", "identity"]),
        ValueError,
    ),
    "non_finite": expect_error(
        lambda: mlp_forward(np.full((2, 4), np.inf), parameters, activations),
        FloatingPointError,
    ),
}
assert len(shape_errors) == 5
print(shape_errors)


### 10. Auditoria dos caches

Cada cache deve corresponder à camada certa. `a_prev` tem a largura de entrada, `z` tem
a largura de saída, e a ativação registrada reproduz o `a_prev` do próximo cache.


In [ ]:
cache_contracts = []
for layer, (cache, expected_activation) in enumerate(zip(caches_batch, activations), start=1):
    expected_z_shape = (x_batch.shape[0], widths[layer])
    contract = {
        "layer": layer,
        "a_prev_shape": cache.a_prev.shape,
        "w_shape": cache.w.shape,
        "b_shape": cache.b.shape,
        "z_shape": cache.z.shape,
        "activation": cache.activation,
    }
    assert cache.a_prev.shape == (x_batch.shape[0], widths[layer - 1])
    assert cache.w.shape == (widths[layer - 1], widths[layer])
    assert cache.b.shape == (widths[layer],)
    assert cache.z.shape == expected_z_shape
    assert cache.activation == expected_activation
    if layer < len(caches_batch):
        reconstructed = activation_forward(cache.z, cache.activation)
        assert np.array_equal(reconstructed, caches_batch[layer].a_prev)
    cache_contracts.append(contract)

print(cache_contracts)


### 11. Memória: lote versus parâmetros

Somamos objetos únicos por identidade para não contar duas vezes a mesma referência. A
parte dinâmica inclui `a_prev` e `z`; parâmetros incluem `w` e `b`. A saída final não é
incluída no cache, portanto deve ser considerada separadamente em um perfil completo.


In [ ]:
def unique_nbytes(arrays: list[np.ndarray]) -> int:
    seen = set()
    total = 0
    for array in arrays:
        if id(array) not in seen:
            seen.add(id(array))
            total += array.nbytes
    return total


def cache_memory(caches: tuple[DenseCache, ...]) -> dict[str, int]:
    dynamic = [array for cache in caches for array in (cache.a_prev, cache.z)]
    params = [array for cache in caches for array in (cache.w, cache.b)]
    return {
        "dynamic_unique_bytes": unique_nbytes(dynamic),
        "parameter_unique_bytes": unique_nbytes(params),
    }


x_small = np.zeros((4, widths[0]), dtype=DTYPE)
x_large = np.zeros((256, widths[0]), dtype=DTYPE)
_, cache_small = mlp_forward(x_small, parameters, activations)
_, cache_large = mlp_forward(x_large, parameters, activations)
memory_small = cache_memory(cache_small)
memory_large = cache_memory(cache_large)
dynamic_ratio = memory_large["dynamic_unique_bytes"] / memory_small["dynamic_unique_bytes"]

assert dynamic_ratio == 64.0
assert memory_large["parameter_unique_bytes"] == memory_small["parameter_unique_bytes"]
print({"batch_4": memory_small, "batch_256": memory_large, "razao_dinamica": dynamic_ratio})


### 12. Curva de memória dinâmica

O gráfico mostra bytes dos arrays dinâmicos preservados no cache para a arquitetura fixa.
O eixo horizontal é o tamanho do lote; o vertical, KiB. Não inclui o processo Python,
temporários, saída final, gradientes ou estado de otimizador.

**Descrição alternativa:** duas curvas em escala logarítmica; a memória dos intermediários
sobe em linha reta com o tamanho do lote, enquanto a linha dos parâmetros permanece horizontal.


In [ ]:
batch_sizes = np.array([1, 4, 16, 64, 256, 1024])
dynamic_bytes = []
parameter_bytes = []
for batch_size in batch_sizes:
    x_memory = np.zeros((int(batch_size), widths[0]), dtype=DTYPE)
    _, memory_caches = mlp_forward(x_memory, parameters, activations)
    memory = cache_memory(memory_caches)
    dynamic_bytes.append(memory["dynamic_unique_bytes"])
    parameter_bytes.append(memory["parameter_unique_bytes"])

dynamic_bytes = np.asarray(dynamic_bytes)
parameter_bytes = np.asarray(parameter_bytes)
assert np.allclose(dynamic_bytes / batch_sizes, dynamic_bytes[0])
assert np.unique(parameter_bytes).size == 1

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(batch_sizes, dynamic_bytes / 1024, marker="o", label="intermediários únicos")
ax.axhline(parameter_bytes[0] / 1024, color="tab:orange", linestyle="--", label="parâmetros")
ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.set_xlabel("tamanho do lote")
ax.set_ylabel("memória auditada (KiB)")
ax.set_title("Cache dinâmico cresce linearmente; parâmetros permanecem fixos")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()


### 13. Contraprova: `frozen` não congela arrays

Usamos um peso de demonstração separado. O cache retém a mesma referência; uma alteração
in place aparece nos dois nomes. Em treino, os parâmetros só poderão ser atualizados após
o backward consumir os caches.


In [ ]:
w_alias = np.array([[1.0], [2.0]], dtype=DTYPE)
b_alias = np.array([0.0], dtype=DTYPE)
x_alias = np.array([[3.0, 4.0]], dtype=DTYPE)
_, alias_cache = dense_forward(x_alias, w_alias, b_alias, "identity")
value_before = float(alias_cache.w[0, 0])
w_alias[0, 0] += 10.0
value_after = float(alias_cache.w[0, 0])

assert alias_cache.w is w_alias
assert value_before == 1.0 and value_after == 11.0
print({
    "mesmo_objeto": alias_cache.w is w_alias,
    "valor_antes": value_before,
    "valor_depois_da_mutacao_externa": value_after,
})


## Checks

Os contratos abaixo consolidam correção algébrica, shapes, vetorização, isolamento de
exemplos, cache, memória e falhas controladas. O teste de aliasing é uma contraprova
intencional, não um comportamento desejado do laço de treinamento.


In [ ]:
checks = {
    "manual_z1": np.array_equal(cache1_manual.z, z1_expected),
    "manual_a1": np.array_equal(a1_manual, a1_expected),
    "manual_logits": np.allclose(logits_manual, logits_expected, atol=1e-15, rtol=0.0),
    "vectorization": vectorization_error < 1e-12,
    "single_example": single_error < 1e-12,
    "permutation": permutation_error < 1e-12,
    "repeatability": repeat_error == 0.0,
    "no_input_mutation": np.array_equal(x_batch, x_before),
    "no_parameter_mutation": all(
        np.array_equal(w, w0) and np.array_equal(b, b0)
        for (w, b), (w0, b0) in zip(parameters, parameters_before)
    ),
    "invalid_cases_rejected": len(shape_errors) == 5,
    "cache_count": len(caches_batch) == len(parameters),
    "cache_chain": all(
        np.array_equal(
            activation_forward(caches_batch[index].z, caches_batch[index].activation),
            caches_batch[index + 1].a_prev,
        )
        for index in range(len(caches_batch) - 1)
    ),
    "dynamic_memory_linear": dynamic_ratio == 64.0,
    "parameter_memory_fixed": memory_large["parameter_unique_bytes"] == memory_small["parameter_unique_bytes"],
    "aliasing_demonstrated": value_before == 1.0 and value_after == 11.0,
}
assert all(checks.values())
print(checks)
print(f"{sum(checks.values())}/{len(checks)} contratos satisfeitos")


## Limites do experimento

- Não calculamos loss, gradientes nem atualização de parâmetros.
- Independência entre linhas vale para as operações usadas; BatchNorm em treino será uma exceção.
- `nbytes` não mede overhead do interpretador, temporários, alocador ou memória do gráfico.
- O número de chamadas `@` não substitui benchmark de hardware.
- A arquitetura é pequena e sintética; os testes validam contratos, não capacidade preditiva.
- O cache didático privilegia uniformidade; implementações otimizadas podem reter menos campos.


## Next Steps

Na **Aula 06 — Losses de regressão e classificação binária**, a saída identidade será
interpretada como predição real ou logit. Derivaremos MSE e binary cross-entropy e
compararemos fórmulas ingênuas com versões numericamente estáveis.
